# Iterative RAG Agent — Telecom Log Analyzer

**RAG pipeline + LLM Agent for telecom log root cause analysis.**

| Component | Library |
|-----------|---------|
| Embeddings | HuggingFace (`all-MiniLM-L6-v2`) |
| Vector Store | ChromaDB |
| LLM | Groq (`llama-3.3-70b-versatile`) |
| Framework | LangChain |
| Web UI | Streamlit |

## Step 1: Install Required Packages

In [15]:
%pip install langchain langchain-community langchain-groq chromadb sentence-transformers groq streamlit -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Imports

In [16]:
import os, re, io, tarfile, zipfile
from groq import Groq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("All imports loaded.")

All imports loaded.


## Step 3: Log Processing Functions

Cleaning, filtering, severity detection, archive extraction — all log types supported.

In [17]:
IMPORTANT_KEYWORDS = [
    "warn", "error", "err", "fail", "timeout", "loss",
    "latency", "delay", "congestion", "critical", "fatal",
    "refused", "rejected", "denied", "abort", "crash",
    "exception", "unreachable", "invalid", "mismatch",
    "drop", "retry", "disconnect", "panic", "oom",
    "not found", "degraded", "down", "offline",
    "rrc release", "ue context release", "crc nok",
    "cell setup", "nok", "ue release", "ngap", "rach",
    "handover", "ho failure", "rlf", "radio link failure",
    "beam failure", "pdu session", "registration reject"
]
SUPPORTED_EXTENSIONS = (".txt", ".log", ".json", ".cfg", ".csv", ".xml", ".html", ".htm")
ARCHIVE_EXTENSIONS = (".tgz", ".tar.gz", ".zip")
ARCHIVE_PATTERNS = [
    "syslog", "messages", "dmesg", "kern.log", "daemon.log",
    "worker", "egate", "alarm", "error", "uec_1", "uec_2",
    "btslog", "rain", "runtime", "gnb", "enb", "cu_cp", "cu_up", "firewall"
]

def clean_line(line):
    line = re.sub(r'\x1b\[[0-9;]*m', '', line)
    line = re.sub(r'^\d{2}:\d{2}:\d{2}\.\d+ \d{2}:\d{2}:\d{2}\.\d+ ', '', line)
    return line.strip()

def is_important(line):
    low = line.lower()
    return any(kw in low for kw in IMPORTANT_KEYWORDS)

def detect_severity(msg):
    low = msg.lower()
    if any(k in low for k in ["error", "data loss", "timeout", "critical", "fatal"]): return "ERROR"
    if "failed" in low: return "FAIL"
    if any(k in low for k in ["warning", "latency", "delay", "congestion"]): return "WARNING"
    return "INFO"

def parse_content(content, filename):
    seen, results = set(), []
    for line in content.splitlines():
        c = clean_line(line)
        if c and len(c) >= 10 and is_important(c) and c not in seen:
            seen.add(c)
            results.append((filename, c))
    return results

def parse_archive(archive_path, archive_name):
    results = []
    try:
        if archive_path.endswith((".tgz", ".tar.gz")):
            with tarfile.open(archive_path, "r:gz") as tar:
                for m in tar.getmembers():
                    if not m.isfile() or m.size > 50*1024*1024: continue
                    if not any(p in m.name.lower() for p in ARCHIVE_PATTERNS): continue
                    f = tar.extractfile(m)
                    if f is None: continue
                    text = f.read().decode("utf-8", errors="ignore")
                    results.extend(parse_content(text, f"{archive_name}/{os.path.basename(m.name)}"))
        elif archive_path.endswith(".zip"):
            with zipfile.ZipFile(archive_path, "r") as zf:
                for name in zf.namelist():
                    if not any(p in name.lower() for p in ARCHIVE_PATTERNS): continue
                    if zf.getinfo(name).file_size > 50*1024*1024: continue
                    text = zf.read(name).decode("utf-8", errors="ignore")
                    results.extend(parse_content(text, f"{archive_name}/{os.path.basename(name)}"))
    except Exception as e:
        print(f"  Archive error {archive_name}: {e}")
    return results

def scan_log_folder(folder):
    all_logs = []
    for root, _, files in os.walk(folder):
        for f in sorted(files):
            fp = os.path.join(root, f)
            rel = os.path.relpath(fp, folder)
            if f.endswith(SUPPORTED_EXTENSIONS):
                with open(fp, "r", encoding="utf-8", errors="ignore") as fh:
                    entries = parse_content(fh.read(), rel)
                if entries: print(f"  {rel}: {len(entries)} entries")
                all_logs.extend(entries)
            elif f.endswith(ARCHIVE_EXTENSIONS):
                entries = parse_archive(fp, rel)
                if entries: print(f"  {rel} (archive): {len(entries)} entries")
                all_logs.extend(entries)
    return all_logs

print("Log processing functions ready.")

Log processing functions ready.


## Step 4: Load Sample Logs & Preview

In [18]:
logs = scan_log_folder("data/logs")
print(f"\nTotal: {len(logs)} entries from {len(set(f for f,_ in logs))} files")

severity_counts = {"ERROR": 0, "FAIL": 0, "WARNING": 0, "INFO": 0}
for _, msg in logs:
    severity_counts[detect_severity(msg)] += 1
print(f"Severity: {severity_counts}\n")

for fname, msg in logs:
    print(f"[{detect_severity(msg):7s}] [{fname}] {msg}")

  log1.txt: 3 entries
  log2.txt: 3 entries
  log3.txt: 3 entries

Total: 9 entries from 3 files
Severity: {'ERROR': 2, 'FAIL': 2, 'WARNING': 3, 'INFO': 2}

[ERROR  ] [log1.txt] UE11 DL Data Loss detected.
[INFO   ] [log1.txt] Packets mismatch threshold exceeded.
[FAIL   ] [log1.txt] Test failed due to network congestion.
[INFO   ] [log2.txt] Registration failure observed.
[ERROR  ] [log2.txt] AMF connection timeout.
[FAIL   ] [log2.txt] UE attach procedure failed.
[WARNING] [log3.txt] High latency detected in network.
[WARNING] [log3.txt] Packet delay exceeded limit.
[WARNING] [log3.txt] Traffic congestion observed.


## Step 5: Build Vector Store (ChromaDB + HuggingFace Embeddings)

Embed all log entries and store in ChromaDB for semantic retrieval.

In [19]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

documents = [
    Document(page_content=msg, metadata={"source": fname, "severity": detect_severity(msg)})
    for fname, msg in logs
]

vectorstore = Chroma.from_documents(documents, embeddings, collection_name="telecom_logs")
print(f"Vector store: {vectorstore._collection.count()} vectors, {len(embeddings.embed_query('test'))}d")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 655.62it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store: 18 vectors, 384d


## Step 6: Test Retriever

Query the vector store — returns the most semantically relevant log entries.

In [20]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

results = retriever.invoke("network timeout failure")
print("Query: 'network timeout failure'\n")
for doc in results:
    print(f"  [{doc.metadata['severity']}] [{doc.metadata['source']}] {doc.page_content}")

Query: 'network timeout failure'

  [ERROR] [log2.txt] AMF connection timeout.
  [ERROR] [log2.txt] AMF connection timeout.
  [WARNING] [log3.txt] Packet delay exceeded limit.
  [WARNING] [log3.txt] Packet delay exceeded limit.
  [FAIL] [log1.txt] Test failed due to network congestion.


## Step 7: Build RAG Chain (Retriever → Prompt → Groq LLM → Answer)

Set your Groq API key below. Get a free key at https://console.groq.com/keys

In [21]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "your_groq_api_key_here")

llm = ChatGroq(groq_api_key=GROQ_API_KEY, model_name="llama-3.3-70b-versatile", max_tokens=500)

rag_prompt = ChatPromptTemplate.from_template(
    """You are a telecom log analysis assistant. Analyze the retrieved log entries and answer.

Format:
- Root Cause: (one line)
- Severity: CRITICAL / HIGH / MEDIUM / LOW
- Details: (explanation with evidence from logs)
- Recommendation: (what to do)

Context (retrieved logs):
{context}

Question: {question}"""
)

def format_docs(docs):
    return "\n".join(f"[{d.metadata.get('severity','?')}] [{d.metadata.get('source','?')}] {d.page_content}" for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)
print("RAG chain ready: Retriever → Prompt → Groq LLM → Answer")

RAG chain ready: Retriever → Prompt → Groq LLM → Answer


## Step 8: Test RAG Query

In [22]:
try:
    answer = rag_chain.invoke("Analyze all errors and find root cause")
    print("Q: Analyze all errors and find root cause\n")
    print(answer)
except Exception as e:
    print(f"LLM Error: {e}")
    print("Update GROQ_API_KEY in Step 7 and re-run.")

LLM Error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}
Update GROQ_API_KEY in Step 7 and re-run.


## Step 9: Generate Streamlit App

Writes `streamlit_app.py` — the full web UI with:
- API key input from sidebar
- File upload for all log formats
- Vector store (ChromaDB) + retriever for every analysis
- Cached embedding model (fast reloads)
- 3 tabs: Single Analysis, Pass vs Fail, Batch

In [23]:
streamlit_code = r'''
import streamlit as st
import os, re, io, tarfile, zipfile, hashlib
from groq import Groq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ─── Constants ───
IMPORTANT_KEYWORDS = [
    "warn", "error", "err", "fail", "timeout", "loss",
    "latency", "delay", "congestion", "critical", "fatal",
    "refused", "rejected", "denied", "abort", "crash",
    "exception", "unreachable", "invalid", "mismatch",
    "drop", "retry", "disconnect", "panic", "oom",
    "not found", "degraded", "down", "offline",
    "rrc release", "ue context release", "crc nok",
    "cell setup", "nok", "ue release", "ngap", "rach",
    "handover", "ho failure", "rlf", "radio link failure",
    "beam failure", "pdu session", "registration reject"
]
ARCHIVE_EXT = (".tgz", ".tar.gz", ".zip")
ARCHIVE_PATTERNS = [
    "syslog", "messages", "dmesg", "kern.log", "daemon.log",
    "worker", "egate", "alarm", "error", "uec_1", "uec_2",
    "btslog", "rain", "runtime", "gnb", "enb", "cu_cp", "cu_up", "firewall"
]
SEV_BADGE = {"ERROR": "\U0001f534", "FAIL": "\U0001f534", "WARNING": "\U0001f7e1", "INFO": "\U0001f535"}

# ─── Cached embedding model (loaded once, reused) ───
@st.cache_resource
def get_embeddings():
    return HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# ─── Log Processing ───
def clean_line(line):
    line = re.sub(r'\x1b\[[0-9;]*m', '', line)
    line = re.sub(r'^\d{2}:\d{2}:\d{2}\.\d+ \d{2}:\d{2}:\d{2}\.\d+ ', '', line)
    return line.strip()

def is_important(line):
    low = line.lower()
    return any(kw in low for kw in IMPORTANT_KEYWORDS)

def detect_severity(msg):
    low = msg.lower()
    if any(k in low for k in ["error", "data loss", "timeout", "critical", "fatal"]): return "ERROR"
    if "failed" in low: return "FAIL"
    if any(k in low for k in ["warning", "latency", "delay", "congestion"]): return "WARNING"
    return "INFO"

def parse_content(content, filename):
    seen, results = set(), []
    for line in content.splitlines():
        c = clean_line(line)
        if c and len(c) >= 10 and is_important(c) and c not in seen:
            seen.add(c)
            results.append((filename, c))
    return results

def parse_archive_bytes(data, name):
    results = []
    try:
        if name.lower().endswith((".tgz", ".tar.gz")):
            with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as tar:
                for m in tar.getmembers():
                    if not m.isfile() or m.size > 50*1024*1024: continue
                    if not any(p in m.name.lower() for p in ARCHIVE_PATTERNS): continue
                    f = tar.extractfile(m)
                    if f: results.extend(parse_content(f.read().decode("utf-8", errors="ignore"), f"{name}/{os.path.basename(m.name)}"))
        elif name.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(data), "r") as zf:
                for n in zf.namelist():
                    if not any(p in n.lower() for p in ARCHIVE_PATTERNS): continue
                    if zf.getinfo(n).file_size > 50*1024*1024: continue
                    results.extend(parse_content(zf.read(n).decode("utf-8", errors="ignore"), f"{name}/{os.path.basename(n)}"))
    except Exception:
        pass
    return results

def process_file(uploaded):
    raw = uploaded.read()
    if uploaded.name.lower().endswith(ARCHIVE_EXT):
        return parse_archive_bytes(raw, uploaded.name)
    return parse_content(raw.decode("utf-8", errors="ignore"), uploaded.name)

# ─── Vector store builder (cached per file content hash) ───
def build_vectorstore(entries, collection_name):
    emb = get_embeddings()
    docs = [Document(page_content=m, metadata={"source": f, "severity": detect_severity(m)}) for f, m in entries]
    vs = Chroma.from_documents(docs, emb, collection_name=collection_name)
    return vs

def format_docs(docs):
    return "\n".join(f"[{d.metadata.get('severity','?')}] [{d.metadata.get('source','?')}] {d.page_content}" for d in docs)

def run_rag(entries, question, api_key, collection_name="analysis"):
    vs = build_vectorstore(entries, collection_name)
    ret = vs.as_retriever(search_kwargs={"k": min(5, len(entries))})
    llm = ChatGroq(groq_api_key=api_key, model_name="llama-3.3-70b-versatile", max_tokens=800)
    prompt = ChatPromptTemplate.from_template(
        "You are a telecom log analyst. Analyze the retrieved logs.\n\n"
        "Format:\n- Root Cause: (one line)\n- Severity: CRITICAL / HIGH / MEDIUM / LOW\n"
        "- Details: (explanation with evidence)\n- Recommendation: (what to do)\n\n"
        "Logs:\n{context}\n\nQuestion: {question}"
    )
    chain = ({"context": ret | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser())
    return chain.invoke(question)

# ─── Page Setup ───
st.set_page_config(page_title="Iterative RAG Agent — Telecom Log Analyzer", page_icon="📡", layout="wide")
st.markdown("# 📡 Iterative RAG Agent — Telecom Log Analyzer")
st.markdown("Upload logs → Vector retrieval → AI root cause analysis")
st.divider()

FILE_TYPES = ["txt","log","json","csv","xml","html","htm","cfg","tgz","gz","zip"]

with st.sidebar:
    st.header("Settings")
    api_key = st.text_input("Groq API Key", type="password", help="https://console.groq.com/keys")
    st.divider()
    st.markdown("**Supported:** `.txt` `.log` `.json` `.csv` `.xml` `.html` `.cfg` `.tgz` `.zip`")

if not api_key:
    st.info("Enter your Groq API key in the sidebar to start.")
    st.stop()

tab1, tab2, tab3 = st.tabs(["Single Analysis", "Pass vs Fail", "Batch Analysis"])

# ═══ TAB 1 ═══
with tab1:
    st.subheader("Upload a log file for AI analysis")
    uploaded = st.file_uploader("Log file", type=FILE_TYPES, key="single")
    query = st.text_input("Question (optional)", placeholder="e.g., Why did UE connection fail?", key="q1")

    if uploaded and st.button("Analyze", type="primary", key="b1"):
        with st.spinner("Processing log..."):
            entries = process_file(uploaded)
        if not entries:
            st.warning("No important entries found in this file.")
        else:
            sev = {"ERROR":0,"FAIL":0,"WARNING":0,"INFO":0}
            for _, m in entries: sev[detect_severity(m)] += 1
            cols = st.columns(4)
            for i,(s,c) in enumerate(sev.items()): cols[i].metric(f"{SEV_BADGE.get(s,'')} {s}", c)

            with st.expander(f"Log entries ({len(entries)} total)"):
                for fn, m in entries[:50]:
                    st.text(f"{SEV_BADGE.get(detect_severity(m),'')} [{fn}] {m}")

            st.divider()
            with st.spinner("AI analyzing with vector retrieval..."):
                q = query if query else "Analyze all errors and find root cause"
                result = run_rag(entries, q, api_key, "single_analysis")
            st.markdown(result)

# ═══ TAB 2 ═══
with tab2:
    st.subheader("Compare PASS and FAIL logs")
    c1, c2 = st.columns(2)
    with c1: pass_file = st.file_uploader("PASS log", type=FILE_TYPES, key="pass")
    with c2: fail_file = st.file_uploader("FAIL log", type=FILE_TYPES, key="fail")

    if pass_file and fail_file and st.button("Compare", type="primary", key="b2"):
        with st.spinner("Comparing with vector retrieval..."):
            pass_entries = process_file(pass_file)
            fail_entries = process_file(fail_file)

            pass_msgs = set(m for _, m in pass_entries)
            common = [(f, m) for f, m in fail_entries if m in pass_msgs]
            fail_only = [(f, m) for f, m in fail_entries if m not in pass_msgs]

            mc = st.columns(3)
            mc[0].metric("PASS entries", len(pass_entries))
            mc[1].metric("Common (ignorable)", len(common))
            mc[2].metric("FAIL-only (critical)", len(fail_only))

            if fail_only:
                with st.expander("Critical Errors (FAIL only)", expanded=True):
                    for _, l in fail_only[:20]: st.text(f"\U0001f534 {l}")
            if common:
                with st.expander("Ignorable (both logs)"):
                    for _, l in common[:20]: st.text(f"\u26aa {l}")

            if fail_only:
                st.divider()
                with st.spinner("AI analyzing FAIL-only errors with vector retrieval..."):
                    result = run_rag(fail_only, "Analyze these FAIL-only errors. What caused the failure? Compare with common errors that appear in both PASS and FAIL.", api_key, "comparison")
                st.markdown(result)
            else:
                st.success("No unique errors in FAIL log — logs are identical.")

# ═══ TAB 3 ═══
with tab3:
    st.subheader("Upload multiple log files")
    batch = st.file_uploader("Log files", type=FILE_TYPES, accept_multiple_files=True, key="batch")
    bq = st.text_input("Question", placeholder="What caused the failure?", key="q3")

    if batch and st.button("Analyze All", type="primary", key="b3"):
        all_entries = []
        for f in batch:
            entries = process_file(f)
            st.text(f"  {f.name}: {len(entries)} entries")
            all_entries.extend(entries)

        st.metric("Total Entries", len(all_entries))
        if all_entries:
            with st.spinner("AI analyzing all files with vector retrieval..."):
                q = bq if bq else "Analyze all errors and find root cause"
                result = run_rag(all_entries, q, api_key, "batch_analysis")
            st.markdown(result)

st.divider()
st.caption("Iterative RAG Agent — LangChain + ChromaDB + HuggingFace + Groq")
'''

with open("streamlit_app.py", "w", encoding="utf-8") as f:
    f.write(streamlit_code)
print("streamlit_app.py written.")

streamlit_app.py written.


## Step 10: Launch Streamlit App

Run the cell below, then open http://localhost:8501 in your browser. Enter your Groq API key in the sidebar and upload log files.

In [ ]:
!streamlit run streamlit_app.py --server.headless true

# AutoRCA — Telecom Log Analyzer (Notebook Version)

**Simple step-by-step RAG pipeline for telecom log root cause analysis.**

Uses: **LangChain** | **ChromaDB** | **HuggingFace Embeddings** | **Groq LLM (Llama 3.3)**

Features:
- Load all log file types (.txt .log .json .csv .xml .html .tgz .zip)
- Auto-detect log type + severity filtering
- Vector store with ChromaDB + HuggingFace embeddings
- RAG-powered root cause analysis via Groq
- Iterative AI investigation (agent asks for more files)
- Pass vs Fail log comparison
- Streamlit web UI at the end

## Step 1: Install Required Packages

In [1]:
%pip install langchain langchain-community langchain-groq chromadb sentence-transformers groq python-dotenv streamlit -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Setup API Key & Imports

In [2]:
import os, re, json, csv, io, tarfile, zipfile
from dotenv import load_dotenv

load_dotenv()

# Your Groq API key — set in .env file or paste here
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "your_groq_api_key_here")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("API Key loaded:", "Yes" if GROQ_API_KEY != "your_groq_api_key_here" else "Set your key above!")

API Key loaded: Yes


## Step 3: Log Cleaning, Filtering & Severity Detection

In [3]:
IMPORTANT_KEYWORDS = [
    "warn", "error", "err", "fail", "timeout", "loss",
    "latency", "delay", "congestion", "critical", "fatal",
    "refused", "rejected", "denied", "abort", "crash",
    "exception", "unreachable", "invalid", "mismatch",
    "drop", "retry", "disconnect", "panic", "oom",
    "not found", "degraded", "down", "offline",
    "rrc release", "ue context release", "crc nok",
    "cell setup", "nok", "ue release", "ngap", "rach",
    "handover", "ho failure", "rlf", "radio link failure",
    "beam failure", "pdu session", "registration reject"
]

SUPPORTED_EXTENSIONS = (".txt", ".log", ".json", ".cfg", ".csv", ".xml", ".html", ".htm")
ARCHIVE_EXTENSIONS = (".tgz", ".tar.gz", ".zip")
ARCHIVE_IMPORTANT_PATTERNS = [
    "syslog", "messages", "dmesg", "kern.log", "daemon.log",
    "worker", "egate", "alarm", "error", "uec_1", "uec_2",
    "btslog", "rain", "runtime", "gnb", "enb", "cu_cp", "cu_up", "firewall"
]

def clean_line(line):
    line = re.sub(r'\x1b\[[0-9;]*m', '', line)
    line = re.sub(r'^\d{2}:\d{2}:\d{2}\.\d+ \d{2}:\d{2}:\d{2}\.\d+ ', '', line)
    return line.strip()

def is_important(line):
    lower = line.lower()
    return any(kw in lower for kw in IMPORTANT_KEYWORDS)

def detect_severity(msg):
    low = msg.lower()
    if any(k in low for k in ["error", "data loss", "timeout", "critical", "fatal"]):
        return "ERROR"
    if "failed" in low:
        return "FAIL"
    if any(k in low for k in ["warning", "latency", "delay", "congestion"]):
        return "WARNING"
    return "INFO"

def detect_log_type(filename, content_sample=""):
    fn = filename.lower()
    if fn.endswith(".json"): return "json_log"
    if fn.endswith(".csv"): return "csv_log"
    if fn.endswith((".xml", ".html", ".htm")): return "xml_log"
    patterns = {
        "egate_console": ["egate_console", "egate"],
        "e2e_test": ["e2e_console", "e2e_output"],
        "syslog": ["syslog", "messages", "kern.log"],
        "uec_log": ["uec_1", "uec_2"],
        "rain_runtime": ["rain", "runtime", "cu_cp", "cu_up"],
        "btslog": ["btslog", "l3_log", "airphone"],
    }
    for log_type, pats in patterns.items():
        if any(p in fn for p in pats):
            return log_type
    return "generic"

print("Log processing functions ready!")

Log processing functions ready!


## Step 4: Load Log Files from data/logs/

Scans all files (txt, log, json, csv, xml, html, cfg) and archives (tgz, zip). Filters only important lines.

In [4]:
LOG_FOLDER = "data/logs"

def read_filtered_logs(filepath, filename):
    """Read a file and return only important lines."""
    seen, results = set(), []
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            cleaned = clean_line(line)
            if cleaned and len(cleaned) >= 10 and is_important(cleaned) and cleaned not in seen:
                seen.add(cleaned)
                results.append((filename, cleaned))
    return results

def extract_archive(archive_path, archive_name):
    """Read important files from tgz/zip archives."""
    results = []
    try:
        if archive_path.endswith((".tgz", ".tar.gz")):
            with tarfile.open(archive_path, "r:gz") as tar:
                for m in tar.getmembers():
                    if not m.isfile() or m.size > 50*1024*1024: continue
                    if not any(p in m.name.lower() for p in ARCHIVE_IMPORTANT_PATTERNS): continue
                    f = tar.extractfile(m)
                    if f is None: continue
                    content = f.read().decode("utf-8", errors="ignore")
                    short = f"{archive_name}/{os.path.basename(m.name)}"
                    seen = set()
                    for line in content.splitlines():
                        c = clean_line(line)
                        if c and len(c) >= 10 and is_important(c) and c not in seen:
                            seen.add(c); results.append((short, c))
        elif archive_path.endswith(".zip"):
            with zipfile.ZipFile(archive_path, "r") as zf:
                for name in zf.namelist():
                    if not any(p in name.lower() for p in ARCHIVE_IMPORTANT_PATTERNS): continue
                    info = zf.getinfo(name)
                    if info.file_size > 50*1024*1024: continue
                    content = zf.read(name).decode("utf-8", errors="ignore")
                    short = f"{archive_name}/{os.path.basename(name)}"
                    seen = set()
                    for line in content.splitlines():
                        c = clean_line(line)
                        if c and len(c) >= 10 and is_important(c) and c not in seen:
                            seen.add(c); results.append((short, c))
    except Exception as e:
        print(f"  Archive error {archive_name}: {e}")
    return results

def scan_log_folder(folder):
    """Scan folder for all log files and archives."""
    all_logs = []
    for root, dirs, files in os.walk(folder):
        for f in sorted(files):
            fp = os.path.join(root, f)
            rel = os.path.relpath(fp, folder)
            if f.endswith(SUPPORTED_EXTENSIONS):
                entries = read_filtered_logs(fp, rel)
                if entries: print(f"  {rel}: {len(entries)} entries")
                all_logs.extend(entries)
            elif f.endswith(ARCHIVE_EXTENSIONS):
                entries = extract_archive(fp, rel)
                if entries: print(f"  {rel} (archive): {len(entries)} entries")
                all_logs.extend(entries)
    return all_logs

print(f"Scanning {LOG_FOLDER}/...\n")
logs = scan_log_folder(LOG_FOLDER)
print(f"\nLoaded {len(logs)} important entries from {len(set(f for f,_ in logs))} files")

Scanning data/logs/...

  log1.txt: 3 entries
  log2.txt: 3 entries
  log3.txt: 3 entries

Loaded 9 important entries from 3 files


## Step 5: Preview Logs with Severity

In [5]:
severity_counts = {"ERROR": 0, "FAIL": 0, "WARNING": 0, "INFO": 0}
for _, msg in logs:
    severity_counts[detect_severity(msg)] += 1

print("Severity Breakdown:", severity_counts)
print(f"\nAll {len(logs)} entries:")
print("-" * 70)
for fname, msg in logs:
    print(f"[{detect_severity(msg):7s}] [{fname}] {msg}")

Severity Breakdown: {'ERROR': 2, 'FAIL': 2, 'WARNING': 3, 'INFO': 2}

All 9 entries:
----------------------------------------------------------------------
[ERROR  ] [log1.txt] UE11 DL Data Loss detected.
[INFO   ] [log1.txt] Packets mismatch threshold exceeded.
[FAIL   ] [log1.txt] Test failed due to network congestion.
[INFO   ] [log2.txt] Registration failure observed.
[ERROR  ] [log2.txt] AMF connection timeout.
[FAIL   ] [log2.txt] UE attach procedure failed.
[WARNING] [log3.txt] High latency detected in network.
[WARNING] [log3.txt] Packet delay exceeded limit.
[WARNING] [log3.txt] Traffic congestion observed.


## Step 6: Create Embeddings & Store in ChromaDB

Using **HuggingFaceEmbeddings** (all-MiniLM-L6-v2) + **ChromaDB** as our vector store.

In [7]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# Create embedding model
print("Loading HuggingFace embedding model...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Convert logs to LangChain Documents
documents = []
for filename, msg in logs:
    documents.append(Document(
        page_content=msg,
        metadata={"source": filename, "severity": detect_severity(msg)}
    ))

# Store in ChromaDB
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="telecom_logs"
)

print(f"ChromaDB vector store created: {vectorstore._collection.count()} vectors")
print(f"Embedding dimension: {len(embeddings.embed_query('test'))}")

Loading HuggingFace embedding model...


C:\Users\anugna\AppData\Local\Temp\ipykernel_22468\3174204811.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
c:\Users\anugna\iterative-rag-project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1337.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.positi

ChromaDB vector store created: 9 vectors
Embedding dimension: 384


## Step 7: Create Retriever & Test Search

Convert ChromaDB to a LangChain retriever — searches for the most relevant log entries for any query.

In [8]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

results = retriever.invoke("network timeout failure")
print("Query: 'network timeout failure'\n")
print(f"Retrieved {len(results)} relevant logs:")
for doc in results:
    sev = doc.metadata.get("severity", "?")
    src = doc.metadata.get("source", "?")
    print(f"  [{sev}] [{src}] {doc.page_content}")

Query: 'network timeout failure'

Retrieved 5 relevant logs:
  [ERROR] [log2.txt] AMF connection timeout.
  [WARNING] [log3.txt] Packet delay exceeded limit.
  [FAIL] [log1.txt] Test failed due to network congestion.
  [WARNING] [log3.txt] High latency detected in network.
  [INFO] [log1.txt] Packets mismatch threshold exceeded.


## Step 8: Initialize Groq LLM & Build RAG Chain

Connect: **Retriever** -> **Prompt** -> **Groq LLM** -> **Answer**

In [9]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Initialize Groq LLM
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile",
    max_tokens=500
)

# RAG prompt template
rag_prompt = ChatPromptTemplate.from_template(
    """You are a telecom log analysis assistant. Analyze the log entries and answer the question.

Give your response in this format:
- Root Cause: (one line)
- Severity: CRITICAL / HIGH / MEDIUM / LOW
- Details: (explanation with evidence from logs)
- Recommendation: (what to do)

Context (retrieved log entries):
{context}

Question: {question}"""
)

# Helper to format retrieved docs
def format_docs(docs):
    return "\n".join(
        f"[{d.metadata.get('severity','?')}] [{d.metadata.get('source','?')}] {d.page_content}"
        for d in docs
    )

# Build the RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready! (Retriever -> Prompt -> Groq LLM -> Answer)")

RAG chain ready! (Retriever -> Prompt -> Groq LLM -> Answer)


## Step 9: Test RAG Chain — Ask Questions About Logs

In [11]:
try:
    answer = rag_chain.invoke("Analyze all errors and find root cause")
    print("Question: Analyze all errors and find root cause\n")
    print(answer)
except Exception as e:
    print(f"LLM Error: {e}")
    print("\nIf you see 'Invalid API Key', update your key in Step 2 and re-run from there.")
    print("Get a free key at: https://console.groq.com/keys")

LLM Error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

If you see 'Invalid API Key', update your key in Step 2 and re-run from there.
Get a free key at: https://console.groq.com/keys


## Step 10: Iterative AI Investigation (Agent Mode)

Same as the full project — AI reads initial errors, asks for more files, digs deeper, and gives root cause.

In [12]:
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)

def ask_llm_direct(messages, max_tokens=500):
    """Send messages to Groq and get response."""
    try:
        resp = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            max_tokens=max_tokens
        )
        return resp.choices[0].message.content
    except Exception as e:
        return f"AI Error: {e}"

def get_logs_from_file(pattern):
    """Get log entries matching a filename pattern."""
    p = pattern.lower()
    return [(f, m) for f, m in logs if p in f.lower()]

def iterative_investigation(query="Analyze all errors and find root cause"):
    """Full iterative AI investigation — same as rag_system.py"""
    available_files = sorted(set(f for f, _ in logs))
    file_list = "\n".join(f"  - {f}" for f in available_files)
    
    error_summary = ""
    for fname, msg in logs[:20]:
        error_summary += f"  - [{detect_severity(msg)}] [{fname}] {msg}\n"
    
    print("=" * 70)
    print("STEP 1: Initial Scan")
    print("=" * 70)
    
    messages = [
        {"role": "system", "content": (
            "You are a telecom log debugging assistant. "
            "Give a brief initial diagnosis, then tell which file(s) to check next. "
            "End with: CHECK_NEXT: filename1, filename2  OR  CHECK_NEXT: DONE"
        )},
        {"role": "user", "content": (
            f"Question: {query}\n\nErrors:\n{error_summary}\nFiles:\n{file_list}"
        )}
    ]
    
    response = ask_llm_direct(messages)
    print(response)
    messages.append({"role": "assistant", "content": response})
    
    for iteration in range(3):
        check_next = None
        for line in response.splitlines():
            if line.strip().upper().startswith("CHECK_NEXT:"):
                check_next = line.split(":", 1)[1].strip()
                break
        
        if not check_next or check_next.upper() == "DONE":
            print(f"\n{'=' * 70}\nInvestigation complete!\n{'=' * 70}")
            break
        
        requested = [f.strip() for f in check_next.split(",")]
        print(f"\n{'=' * 70}\nSTEP {iteration+2}: AI requested: {requested}\n{'=' * 70}")
        
        deep_logs = ""
        for pat in requested:
            matched = get_logs_from_file(pat)
            if matched:
                deep_logs += f"\n[{pat}] ({len(matched)} entries):\n"
                for fn, msg in matched[:20]:
                    deep_logs += f"  - {msg}\n"
            else:
                deep_logs += f"\n[{pat}]: Not found.\n"
        
        messages.append({"role": "user", "content": (
            f"Logs from requested files:\n{deep_logs}\n\n"
            "Give deeper analysis. End with CHECK_NEXT: files or CHECK_NEXT: DONE"
        )})
        
        response = ask_llm_direct(messages)
        print(response)
        messages.append({"role": "assistant", "content": response})

# Run it!
try:
    iterative_investigation()
except Exception as e:
    print(f"Error: {e}\nUpdate your API key in Step 2 if needed.")

STEP 1: Initial Scan
AI Error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

Investigation complete!


## Step 11: Pass vs Fail Comparison

Compare a PASS log and FAIL log — finds errors unique to the failure, ignoring common noise.

In [13]:
def compare_pass_fail(pass_path, fail_path):
    """Compare PASS and FAIL logs to find unique failure errors."""
    with open(pass_path, "r", encoding="utf-8", errors="ignore") as f:
        pass_content = f.read()
    with open(fail_path, "r", encoding="utf-8", errors="ignore") as f:
        fail_content = f.read()
    
    pass_lines = set(clean_line(l) for l in pass_content.splitlines() if len(clean_line(l)) >= 10)
    
    common_errors, fail_only_errors = [], []
    fail_set = set()
    for line in fail_content.splitlines():
        c = clean_line(line)
        if not c or len(c) < 10 or c in fail_set: continue
        fail_set.add(c)
        if is_important(c):
            if c in pass_lines:
                common_errors.append(c)
            else:
                fail_only_errors.append(c)
    
    print(f"PASS: {os.path.basename(pass_path)} ({len(pass_lines)} lines)")
    print(f"FAIL: {os.path.basename(fail_path)} ({len(fail_set)} lines)")
    print(f"\nIgnorable (in both): {len(common_errors)}")
    for l in common_errors[:5]: print(f"  {l}")
    print(f"\nCritical (FAIL only): {len(fail_only_errors)}")
    for l in fail_only_errors[:10]: print(f"  {l}")
    
    # LLM analysis
    common_str = "\n".join(f"  - {l}" for l in common_errors[:15]) or "  (none)"
    fail_str = "\n".join(f"  - {l}" for l in fail_only_errors[:20]) or "  (none)"
    messages = [
        {"role": "system", "content": "You are a telecom log analyst. Compare PASS vs FAIL logs. Focus on FAIL-only errors."},
        {"role": "user", "content": f"Common errors (ignorable):\n{common_str}\n\nFAIL-only errors:\n{fail_str}"}
    ]
    print(f"\n{'='*70}\nAI Comparison Analysis:\n{'='*70}")
    print(ask_llm_direct(messages, max_tokens=600))

# Example: compare_pass_fail("data/logs/pass.log", "data/logs/fail.log")
print("Usage: compare_pass_fail('path/to/pass.log', 'path/to/fail.log')")
print("This will be available in the Streamlit UI below too.")

Usage: compare_pass_fail('path/to/pass.log', 'path/to/fail.log')
This will be available in the Streamlit UI below too.


## Step 12: Generate & Launch Streamlit Web App

This creates a Streamlit app with the same features:
- Upload log files (all formats)
- Auto-detect log type & severity
- RAG-powered AI analysis with Groq
- Pass vs Fail comparison
- Enter API key from sidebar

In [ ]:
streamlit_code = r'''
import streamlit as st
import os, re, json, csv, io, tarfile, zipfile
from groq import Groq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ─── Constants ───
IMPORTANT_KEYWORDS = [
    "warn", "error", "err", "fail", "timeout", "loss",
    "latency", "delay", "congestion", "critical", "fatal",
    "refused", "rejected", "denied", "abort", "crash",
    "exception", "unreachable", "invalid", "mismatch",
    "drop", "retry", "disconnect", "panic", "oom",
    "not found", "degraded", "down", "offline",
    "rrc release", "ue context release", "crc nok",
    "cell setup", "nok", "ue release", "ngap", "rach",
    "handover", "ho failure", "rlf", "radio link failure",
    "beam failure", "pdu session", "registration reject"
]
SUPPORTED_EXT = (".txt", ".log", ".json", ".cfg", ".csv", ".xml", ".html", ".htm")
ARCHIVE_EXT = (".tgz", ".tar.gz", ".zip")
ARCHIVE_PATTERNS = [
    "syslog", "messages", "dmesg", "kern.log", "daemon.log",
    "worker", "egate", "alarm", "error", "uec_1", "uec_2",
    "btslog", "rain", "runtime", "gnb", "enb", "cu_cp", "cu_up", "firewall"
]

# ─── Utility Functions ───
def clean_line(line):
    line = re.sub(r'\x1b\[[0-9;]*m', '', line)
    line = re.sub(r'^\d{2}:\d{2}:\d{2}\.\d+ \d{2}:\d{2}:\d{2}\.\d+ ', '', line)
    return line.strip()

def is_important(line):
    low = line.lower()
    return any(kw in low for kw in IMPORTANT_KEYWORDS)

def detect_severity(msg):
    low = msg.lower()
    if any(k in low for k in ["error", "data loss", "timeout", "critical", "fatal"]): return "ERROR"
    if "failed" in low: return "FAIL"
    if any(k in low for k in ["warning", "latency", "delay", "congestion"]): return "WARNING"
    return "INFO"

def parse_content(content, filename):
    seen, results = set(), []
    for line in content.splitlines():
        c = clean_line(line)
        if c and len(c) >= 10 and is_important(c) and c not in seen:
            seen.add(c)
            results.append((filename, c))
    return results

def parse_archive_bytes(archive_bytes, archive_name):
    results = []
    try:
        if archive_name.lower().endswith((".tgz", ".tar.gz")):
            with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as tar:
                for m in tar.getmembers():
                    if not m.isfile() or m.size > 50*1024*1024: continue
                    if not any(p in m.name.lower() for p in ARCHIVE_PATTERNS): continue
                    f = tar.extractfile(m)
                    if f is None: continue
                    text = f.read().decode("utf-8", errors="ignore")
                    short = f"{archive_name}/{os.path.basename(m.name)}"
                    results.extend(parse_content(text, short))
        elif archive_name.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(archive_bytes), "r") as zf:
                for name in zf.namelist():
                    if not any(p in name.lower() for p in ARCHIVE_PATTERNS): continue
                    info = zf.getinfo(name)
                    if info.file_size > 50*1024*1024: continue
                    text = zf.read(name).decode("utf-8", errors="ignore")
                    short = f"{archive_name}/{os.path.basename(name)}"
                    results.extend(parse_content(text, short))
    except Exception:
        pass
    return results

def process_file(uploaded_file):
    fname = uploaded_file.name
    fbytes = uploaded_file.read()
    if fname.lower().endswith(ARCHIVE_EXT):
        return parse_archive_bytes(fbytes, fname)
    content = fbytes.decode("utf-8", errors="ignore")
    return parse_content(content, fname)

SEV_BADGE = {"ERROR": "🔴", "FAIL": "🔴", "WARNING": "🟡", "INFO": "🔵"}

# ─── Page Config ───
st.set_page_config(page_title="AutoRCA — Telecom Log Analyzer", page_icon="🔍", layout="wide")
st.markdown("# 🔍 AutoRCA — Telecom Log Analyzer")
st.markdown("**RAG + LLM Agent** | Upload logs → AI finds root cause")
st.divider()

# ─── Sidebar ───
with st.sidebar:
    st.header("⚙️ Settings")
    api_key = st.text_input("Groq API Key", type="password", help="https://console.groq.com/keys")
    if api_key:
        os.environ["GROQ_API_KEY"] = api_key
    st.divider()
    st.markdown("**Supported:** `.txt` `.log` `.json` `.csv` `.xml` `.html` `.cfg` `.tgz` `.zip`")

groq_key = os.environ.get("GROQ_API_KEY", "")
if not groq_key:
    st.warning("Enter your Groq API key in the sidebar to start.")

# ─── Tabs ───
tab1, tab2, tab3 = st.tabs(["📊 Single Analysis", "🔄 Pass vs Fail", "📁 Batch Analysis"])

# ═══ TAB 1: Single Log Analysis ═══
with tab1:
    st.subheader("Upload a log file for AI root cause analysis")
    uploaded = st.file_uploader("Upload log file", type=["txt","log","json","csv","xml","html","htm","cfg","tgz","gz","zip"], key="single")
    query = st.text_input("Custom question (optional)", placeholder="e.g., Why did UE connection fail?", key="q1")

    if uploaded and st.button("Analyze", type="primary", key="b1"):
        if not groq_key:
            st.error("Enter API key in sidebar.")
        else:
            with st.spinner("Processing..."):
                entries = process_file(uploaded)
            
            col1, col2 = st.columns(2)
            col1.metric("File", uploaded.name)
            col2.metric("Filtered Entries", len(entries))
            
            if not entries:
                st.warning("No important entries found.")
            else:
                sev_counts = {"ERROR":0,"FAIL":0,"WARNING":0,"INFO":0}
                for _, m in entries:
                    sev_counts[detect_severity(m)] += 1
                cols = st.columns(4)
                for i,(s,c) in enumerate(sev_counts.items()):
                    cols[i].metric(f"{SEV_BADGE.get(s,'')} {s}", c)
                
                with st.expander("Preview entries"):
                    for fn, m in entries[:50]:
                        st.text(f"{SEV_BADGE.get(detect_severity(m),'')} [{fn}] {m}")
                
                st.divider()
                st.subheader("🤖 AI Analysis")
                with st.spinner("AI analyzing..."):
                    # Build ChromaDB from uploaded entries
                    emb = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
                    docs = [Document(page_content=m, metadata={"source":f,"severity":detect_severity(m)}) for f,m in entries]
                    vs = Chroma.from_documents(docs, emb, collection_name="upload_analysis")
                    ret = vs.as_retriever(search_kwargs={"k":5})
                    
                    llm = ChatGroq(groq_api_key=groq_key, model_name="llama-3.3-70b-versatile", max_tokens=800)
                    prompt = ChatPromptTemplate.from_template(
                        "You are a telecom log analyst. Analyze logs and give Root Cause, Severity, Details, Recommendation.\n\nLogs:\n{context}\n\nQuestion: {question}"
                    )
                    def fmt(docs): return "\n".join(f"[{d.metadata.get('severity','?')}] [{d.metadata.get('source','?')}] {d.page_content}" for d in docs)
                    chain = ({"context": ret | fmt, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser())
                    
                    q = query if query else "Analyze all errors and find root cause"
                    result = chain.invoke(q)
                
                st.markdown(result)

# ═══ TAB 2: Pass vs Fail ═══
with tab2:
    st.subheader("Compare PASS and FAIL logs")
    col_p, col_f = st.columns(2)
    with col_p:
        pass_file = st.file_uploader("PASS log", type=["txt","log","json","csv","xml","html","htm","cfg","tgz","gz","zip"], key="pass")
    with col_f:
        fail_file = st.file_uploader("FAIL log", type=["txt","log","json","csv","xml","html","htm","cfg","tgz","gz","zip"], key="fail")
    
    if pass_file and fail_file and st.button("Compare", type="primary", key="b2"):
        if not groq_key:
            st.error("Enter API key in sidebar.")
        else:
            with st.spinner("Comparing..."):
                pc = pass_file.read().decode("utf-8", errors="ignore")
                fc = fail_file.read().decode("utf-8", errors="ignore")
                
                pass_lines = set(clean_line(l) for l in pc.splitlines() if len(clean_line(l)) >= 10)
                common_errs, fail_only = [], []
                fail_set = set()
                for line in fc.splitlines():
                    c = clean_line(line)
                    if not c or len(c) < 10 or c in fail_set: continue
                    fail_set.add(c)
                    if is_important(c):
                        if c in pass_lines: common_errs.append(c)
                        else: fail_only.append(c)
                
                c1,c2,c3 = st.columns(3)
                c1.metric("PASS lines", len(pass_lines))
                c2.metric("Common (ignorable)", len(common_errs))
                c3.metric("FAIL-only (critical)", len(fail_only))
                
                if fail_only:
                    with st.expander("Critical Errors (FAIL only)", expanded=True):
                        for l in fail_only[:20]: st.text(f"🔴 {l}")
                if common_errs:
                    with st.expander("Ignorable Errors (both)"):
                        for l in common_errs[:20]: st.text(f"⚪ {l}")
                
                st.divider()
                client = Groq(api_key=groq_key)
                msgs = [
                    {"role":"system","content":"You are a telecom log analyst. Compare PASS vs FAIL. Focus on FAIL-only errors. Give Root Cause, Severity, Details, Recommendation."},
                    {"role":"user","content":f"Common:\n{chr(10).join(common_errs[:15])}\n\nFAIL-only:\n{chr(10).join(fail_only[:20])}"}
                ]
                resp = client.chat.completions.create(model="llama-3.3-70b-versatile", messages=msgs, max_tokens=800)
                st.markdown(resp.choices[0].message.content)

# ═══ TAB 3: Batch Analysis ═══
with tab3:
    st.subheader("Upload multiple log files")
    batch = st.file_uploader("Upload files", type=["txt","log","json","csv","xml","html","htm","cfg","tgz","gz","zip"], accept_multiple_files=True, key="batch")
    bq = st.text_input("Custom question", placeholder="What caused the failure?", key="q3")
    
    if batch and st.button("Analyze All", type="primary", key="b3"):
        if not groq_key:
            st.error("Enter API key in sidebar.")
        else:
            all_entries = []
            for f in batch:
                entries = process_file(f)
                st.text(f"  {f.name}: {len(entries)} entries")
                all_entries.extend(entries)
            
            st.metric("Total Entries", len(all_entries))
            
            if all_entries:
                with st.spinner("AI analyzing all files..."):
                    emb = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
                    docs = [Document(page_content=m, metadata={"source":f,"severity":detect_severity(m)}) for f,m in all_entries]
                    vs = Chroma.from_documents(docs, emb, collection_name="batch_analysis")
                    ret = vs.as_retriever(search_kwargs={"k":5})
                    
                    llm = ChatGroq(groq_api_key=groq_key, model_name="llama-3.3-70b-versatile", max_tokens=800)
                    prompt = ChatPromptTemplate.from_template(
                        "You are a telecom log analyst. Analyze logs and give Root Cause, Severity, Details, Recommendation.\n\nLogs:\n{context}\n\nQuestion: {question}"
                    )
                    def fmt(docs): return "\n".join(f"[{d.metadata.get('severity','?')}] [{d.metadata.get('source','?')}] {d.page_content}" for d in docs)
                    chain = ({"context": ret | fmt, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser())
                    
                    q = bq if bq else "Analyze all errors and find root cause"
                    result = chain.invoke(q)
                st.markdown(result)

st.divider()
st.caption("AutoRCA — Powered by LangChain + ChromaDB + HuggingFace + Groq LLM")
'''

# Write the Streamlit app file
app_path = os.path.join(os.getcwd(), "streamlit_app.py")
with open(app_path, "w", encoding="utf-8") as f:
    f.write(streamlit_code)
print(f"Streamlit app written to: {app_path}")

In [14]:
streamlit_code = r'''
import streamlit as st
import os, re, json, csv, io, tarfile, zipfile
from groq import Groq
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ─── Constants ───
IMPORTANT_KEYWORDS = [
    "warn", "error", "err", "fail", "timeout", "loss",
    "latency", "delay", "congestion", "critical", "fatal",
    "refused", "rejected", "denied", "abort", "crash",
    "exception", "unreachable", "invalid", "mismatch",
    "drop", "retry", "disconnect", "panic", "oom",
    "not found", "degraded", "down", "offline",
    "rrc release", "ue context release", "crc nok",
    "cell setup", "nok", "ue release", "ngap", "rach",
    "handover", "ho failure", "rlf", "radio link failure",
    "beam failure", "pdu session", "registration reject"
]
SUPPORTED_EXT = (".txt", ".log", ".json", ".cfg", ".csv", ".xml", ".html", ".htm")
ARCHIVE_EXT = (".tgz", ".tar.gz", ".zip")
ARCHIVE_PATTERNS = [
    "syslog", "messages", "dmesg", "kern.log", "daemon.log",
    "worker", "egate", "alarm", "error", "uec_1", "uec_2",
    "btslog", "rain", "runtime", "gnb", "enb", "cu_cp", "cu_up", "firewall"
]

# ─── Utility Functions ───
def clean_line(line):
    line = re.sub(r'\x1b\[[0-9;]*m', '', line)
    line = re.sub(r'^\d{2}:\d{2}:\d{2}\.\d+ \d{2}:\d{2}:\d{2}\.\d+ ', '', line)
    return line.strip()

def is_important(line):
    low = line.lower()
    return any(kw in low for kw in IMPORTANT_KEYWORDS)

def detect_severity(msg):
    low = msg.lower()
    if any(k in low for k in ["error", "data loss", "timeout", "critical", "fatal"]): return "ERROR"
    if "failed" in low: return "FAIL"
    if any(k in low for k in ["warning", "latency", "delay", "congestion"]): return "WARNING"
    return "INFO"

def parse_content(content, filename):
    seen, results = set(), []
    for line in content.splitlines():
        c = clean_line(line)
        if c and len(c) >= 10 and is_important(c) and c not in seen:
            seen.add(c)
            results.append((filename, c))
    return results

def parse_archive_bytes(archive_bytes, archive_name):
    results = []
    try:
        if archive_name.lower().endswith((".tgz", ".tar.gz")):
            with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as tar:
                for m in tar.getmembers():
                    if not m.isfile() or m.size > 50*1024*1024: continue
                    if not any(p in m.name.lower() for p in ARCHIVE_PATTERNS): continue
                    f = tar.extractfile(m)
                    if f is None: continue
                    text = f.read().decode("utf-8", errors="ignore")
                    short = f"{archive_name}/{os.path.basename(m.name)}"
                    results.extend(parse_content(text, short))
        elif archive_name.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(archive_bytes), "r") as zf:
                for name in zf.namelist():
                    if not any(p in name.lower() for p in ARCHIVE_PATTERNS): continue
                    info = zf.getinfo(name)
                    if info.file_size > 50*1024*1024: continue
                    text = zf.read(name).decode("utf-8", errors="ignore")
                    short = f"{archive_name}/{os.path.basename(name)}"
                    results.extend(parse_content(text, short))
    except Exception:
        pass
    return results

def process_file(uploaded_file):
    fname = uploaded_file.name
    fbytes = uploaded_file.read()
    if fname.lower().endswith(ARCHIVE_EXT):
        return parse_archive_bytes(fbytes, fname)
    content = fbytes.decode("utf-8", errors="ignore")
    return parse_content(content, fname)

SEV_BADGE = {"ERROR": "🔴", "FAIL": "🔴", "WARNING": "🟡", "INFO": "🔵"}

# ─── Page Config ───
st.set_page_config(page_title="AutoRCA — Telecom Log Analyzer", page_icon="🔍", layout="wide")
st.markdown("# 🔍 AutoRCA — Telecom Log Analyzer")
st.markdown("**RAG + LLM Agent** | Upload logs → AI finds root cause")
st.divider()

# ─── Sidebar ───
with st.sidebar:
    st.header("⚙️ Settings")
    api_key = st.text_input("Groq API Key", type="password", help="https://console.groq.com/keys")
    if api_key:
        os.environ["GROQ_API_KEY"] = api_key
    st.divider()
    st.markdown("**Supported:** `.txt` `.log` `.json` `.csv` `.xml` `.html` `.cfg` `.tgz` `.zip`")

groq_key = os.environ.get("GROQ_API_KEY", "")
if not groq_key:
    st.warning("Enter your Groq API key in the sidebar to start.")

# ─── Tabs ───
tab1, tab2, tab3 = st.tabs(["📊 Single Analysis", "🔄 Pass vs Fail", "📁 Batch Analysis"])

# ═══ TAB 1: Single Log Analysis ═══
with tab1:
    st.subheader("Upload a log file for AI root cause analysis")
    uploaded = st.file_uploader("Upload log file", type=["txt","log","json","csv","xml","html","htm","cfg","tgz","gz","zip"], key="single")
    query = st.text_input("Custom question (optional)", placeholder="e.g., Why did UE connection fail?", key="q1")

    if uploaded and st.button("Analyze", type="primary", key="b1"):
        if not groq_key:
            st.error("Enter API key in sidebar.")
        else:
            with st.spinner("Processing..."):
                entries = process_file(uploaded)
            
            col1, col2 = st.columns(2)
            col1.metric("File", uploaded.name)
            col2.metric("Filtered Entries", len(entries))
            
            if not entries:
                st.warning("No important entries found.")
            else:
                sev_counts = {"ERROR":0,"FAIL":0,"WARNING":0,"INFO":0}
                for _, m in entries:
                    sev_counts[detect_severity(m)] += 1
                cols = st.columns(4)
                for i,(s,c) in enumerate(sev_counts.items()):
                    cols[i].metric(f"{SEV_BADGE.get(s,'')} {s}", c)
                
                with st.expander("Preview entries"):
                    for fn, m in entries[:50]:
                        st.text(f"{SEV_BADGE.get(detect_severity(m),'')} [{fn}] {m}")
                
                st.divider()
                st.subheader("🤖 AI Analysis")
                with st.spinner("AI analyzing..."):
                    # Build ChromaDB from uploaded entries
                    emb = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
                    docs = [Document(page_content=m, metadata={"source":f,"severity":detect_severity(m)}) for f,m in entries]
                    vs = Chroma.from_documents(docs, emb, collection_name="upload_analysis")
                    ret = vs.as_retriever(search_kwargs={"k":5})
                    
                    llm = ChatGroq(groq_api_key=groq_key, model_name="llama-3.3-70b-versatile", max_tokens=800)
                    prompt = ChatPromptTemplate.from_template(
                        "You are a telecom log analyst. Analyze logs and give Root Cause, Severity, Details, Recommendation.\n\nLogs:\n{context}\n\nQuestion: {question}"
                    )
                    def fmt(docs): return "\n".join(f"[{d.metadata.get('severity','?')}] [{d.metadata.get('source','?')}] {d.page_content}" for d in docs)
                    chain = ({"context": ret | fmt, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser())
                    
                    q = query if query else "Analyze all errors and find root cause"
                    result = chain.invoke(q)
                
                st.markdown(result)

# ═══ TAB 2: Pass vs Fail ═══
with tab2:
    st.subheader("Compare PASS and FAIL logs")
    col_p, col_f = st.columns(2)
    with col_p:
        pass_file = st.file_uploader("PASS log", type=["txt","log","json","csv","xml","html","htm","cfg","tgz","gz","zip"], key="pass")
    with col_f:
        fail_file = st.file_uploader("FAIL log", type=["txt","log","json","csv","xml","html","htm","cfg","tgz","gz","zip"], key="fail")
    
    if pass_file and fail_file and st.button("Compare", type="primary", key="b2"):
        if not groq_key:
            st.error("Enter API key in sidebar.")
        else:
            with st.spinner("Comparing..."):
                pc = pass_file.read().decode("utf-8", errors="ignore")
                fc = fail_file.read().decode("utf-8", errors="ignore")
                
                pass_lines = set(clean_line(l) for l in pc.splitlines() if len(clean_line(l)) >= 10)
                common_errs, fail_only = [], []
                fail_set = set()
                for line in fc.splitlines():
                    c = clean_line(line)
                    if not c or len(c) < 10 or c in fail_set: continue
                    fail_set.add(c)
                    if is_important(c):
                        if c in pass_lines: common_errs.append(c)
                        else: fail_only.append(c)
                
                c1,c2,c3 = st.columns(3)
                c1.metric("PASS lines", len(pass_lines))
                c2.metric("Common (ignorable)", len(common_errs))
                c3.metric("FAIL-only (critical)", len(fail_only))
                
                if fail_only:
                    with st.expander("Critical Errors (FAIL only)", expanded=True):
                        for l in fail_only[:20]: st.text(f"🔴 {l}")
                if common_errs:
                    with st.expander("Ignorable Errors (both)"):
                        for l in common_errs[:20]: st.text(f"⚪ {l}")
                
                st.divider()
                client = Groq(api_key=groq_key)
                msgs = [
                    {"role":"system","content":"You are a telecom log analyst. Compare PASS vs FAIL. Focus on FAIL-only errors. Give Root Cause, Severity, Details, Recommendation."},
                    {"role":"user","content":f"Common:\n{chr(10).join(common_errs[:15])}\n\nFAIL-only:\n{chr(10).join(fail_only[:20])}"}
                ]
                resp = client.chat.completions.create(model="llama-3.3-70b-versatile", messages=msgs, max_tokens=800)
                st.markdown(resp.choices[0].message.content)

# ═══ TAB 3: Batch Analysis ═══
with tab3:
    st.subheader("Upload multiple log files")
    batch = st.file_uploader("Upload files", type=["txt","log","json","csv","xml","html","htm","cfg","tgz","gz","zip"], accept_multiple_files=True, key="batch")
    bq = st.text_input("Custom question", placeholder="What caused the failure?", key="q3")
    
    if batch and st.button("Analyze All", type="primary", key="b3"):
        if not groq_key:
            st.error("Enter API key in sidebar.")
        else:
            all_entries = []
            for f in batch:
                entries = process_file(f)
                st.text(f"  {f.name}: {len(entries)} entries")
                all_entries.extend(entries)
            
            st.metric("Total Entries", len(all_entries))
            
            if all_entries:
                with st.spinner("AI analyzing all files..."):
                    emb = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
                    docs = [Document(page_content=m, metadata={"source":f,"severity":detect_severity(m)}) for f,m in all_entries]
                    vs = Chroma.from_documents(docs, emb, collection_name="batch_analysis")
                    ret = vs.as_retriever(search_kwargs={"k":5})
                    
                    llm = ChatGroq(groq_api_key=groq_key, model_name="llama-3.3-70b-versatile", max_tokens=800)
                    prompt = ChatPromptTemplate.from_template(
                        "You are a telecom log analyst. Analyze logs and give Root Cause, Severity, Details, Recommendation.\n\nLogs:\n{context}\n\nQuestion: {question}"
                    )
                    def fmt(docs): return "\n".join(f"[{d.metadata.get('severity','?')}] [{d.metadata.get('source','?')}] {d.page_content}" for d in docs)
                    chain = ({"context": ret | fmt, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser())
                    
                    q = bq if bq else "Analyze all errors and find root cause"
                    result = chain.invoke(q)
                st.markdown(result)

st.divider()
st.caption("AutoRCA — Powered by LangChain + ChromaDB + HuggingFace + Groq LLM")
'''

# Write the Streamlit app file
app_path = os.path.join(os.getcwd(), "streamlit_app.py")
with open(app_path, "w", encoding="utf-8") as f:
    f.write(streamlit_code)
print(f"Streamlit app written to: {app_path}")

Streamlit app written to: c:\Users\anugna\iterative-rag-project\streamlit_app.py


## Step 13: Launch Streamlit App

Run the cell below to start the web UI. Open the URL shown in the output (usually http://localhost:8501).

In [ ]:
!streamlit run streamlit_app.py --server.headless true

In [ ]:
# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Test it
results = retriever.invoke("network timeout failure")
print("Query: 'network timeout failure'\n")
print(f"Retrieved {len(results)} relevant logs:")
for doc in results:
    sev = doc.metadata.get("severity", "?")
    src = doc.metadata.get("source", "?")
    print(f"  [{sev}] [{src}] {doc.page_content}")

# AutoRCA — Telecom Log Analyzer (Notebook Version)

This notebook is a **simple step-by-step version** of our Automated Root Cause Analysis system.

**What it does (same as the full project):**
- Loads telecom log files (`.txt`, `.log`, `.json`, `.csv`, `.xml`, `.html`, `.tgz`, `.zip`)
- Cleans and filters important log lines (errors, warnings, failures)
- Auto-detects log type (egate, syslog, e2e test, sosreport, etc.)
- Creates embeddings using **sentence-transformers** (all-MiniLM-L6-v2)
- Stores vectors in **FAISS** for fast similarity search
- Uses **Groq LLM** (Llama 3.3) to analyze logs and find root cause
- Supports **iterative investigation** — AI asks for more files if needed
- Supports **Pass vs Fail comparison**

## Step 1: Install Required Packages

In [ ]:
!pip install faiss-cpu sentence-transformers numpy groq python-dotenv

## Step 2: Set Up API Key

Set your **Groq API key**. Get a free one from https://console.groq.com/keys

In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq

# Load from .env file if it exists
load_dotenv()

# Set your Groq API key (or put it in .env file as GROQ_API_KEY=gsk_xxx)
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "your_groq_api_key_here")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Initialize Groq client
groq_client = Groq(api_key=GROQ_API_KEY)
print("Groq client ready!" if GROQ_API_KEY != "your_groq_api_key_here" else "WARNING: Set your API key above!")

## Step 3: Define Log Cleaning and Filtering Functions

These functions clean raw log lines (remove ANSI codes, timestamps) and filter only **important** lines containing error/warning keywords — same as our main project.

In [ ]:
import re

# Keywords that indicate important log lines
IMPORTANT_KEYWORDS = [
    "warn", "error", "err", "fail", "timeout", "loss",
    "latency", "delay", "congestion", "critical", "fatal",
    "refused", "rejected", "denied", "abort", "crash",
    "exception", "unreachable", "invalid", "mismatch",
    "obsolete", "drop", "retry", "disconnect",
    "panic", "oom", "killed", "segfault", "oops",
    "not found", "degraded", "down", "offline",
    # Telecom-specific
    "rrc release", "ue context release", "crc nok",
    "cell setup", "not received", "nok", "ue release",
    "ngap", "rach", "handover", "ho failure",
    "rlf", "radio link failure", "beam failure",
    "pdu session", "registration reject", "attach reject"
]

def clean_line(line):
    """Remove ANSI escape codes and timestamps from a log line."""
    line = re.sub(r'\x1b\[[0-9;]*m', '', line)
    line = re.sub(r'^\d{2}:\d{2}:\d{2}\.\d+ \d{2}:\d{2}:\d{2}\.\d+ ', '', line)
    return line.strip()

def is_important(line):
    """Check if a log line contains error/warning keywords."""
    lower = line.lower()
    return any(kw in lower for kw in IMPORTANT_KEYWORDS)

def detect_severity(log_message):
    """Classify severity: ERROR, FAIL, WARNING, or INFO."""
    log_lower = log_message.lower()
    if any(kw in log_lower for kw in ["error", "data loss", "timeout", "critical", "fatal"]):
        return "ERROR"
    elif "failed" in log_lower:
        return "FAIL"
    elif any(kw in log_lower for kw in ["warning", "latency", "delay", "congestion"]):
        return "WARNING"
    return "INFO"

print("Log cleaning and filtering functions ready!")

## Step 4: Load Log Files from `data/logs/` Folder

Scans all log files (`.txt`, `.log`, `.json`, `.csv`, `.xml`, `.html`, `.cfg`) **and** archives (`.tgz`, `.zip`) from the `data/logs/` folder. Filters to keep only important lines.

> **To add your own logs**: just drop files into the `data/logs/` folder and re-run this cell.

In [ ]:
import os
import json
import csv
import io
import tarfile
import zipfile

LOG_FOLDER = "data/logs"
SUPPORTED_EXTENSIONS = (".txt", ".log", ".json", ".cfg", ".csv", ".xml", ".html", ".htm")
ARCHIVE_EXTENSIONS = (".tgz", ".tar.gz", ".zip")

# Patterns to look for inside archives (sosreport, etc.)
ARCHIVE_IMPORTANT_PATTERNS = [
    "syslog", "messages", "dmesg", "journalctl", "kern.log",
    "daemon.log", "boot.log", "auth.log", "cron.log",
    "worker", "egate", "alarm", "error",
    "uec_1", "uec_2", "btslog", "rain", "runtime",
    "gnb", "enb", "cu_cp", "cu_up", "du_", "firewall"
]

def read_filtered_logs(filepath, filename):
    """Read a log file, keep only important unique lines."""
    seen = set()
    results = []
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            cleaned = clean_line(line)
            if not cleaned or len(cleaned) < 10:
                continue
            if is_important(cleaned) and cleaned not in seen:
                seen.add(cleaned)
                results.append((filename, cleaned))
    return results

def extract_and_read_archive(archive_path, archive_name):
    """Extract important files from .tgz/.zip archives."""
    results = []
    try:
        if archive_path.endswith((".tgz", ".tar.gz")):
            with tarfile.open(archive_path, "r:gz") as tar:
                for member in tar.getmembers():
                    if not member.isfile() or member.size > 50 * 1024 * 1024:
                        continue
                    if not any(p in member.name.lower() for p in ARCHIVE_IMPORTANT_PATTERNS):
                        continue
                    try:
                        f = tar.extractfile(member)
                        if f is None:
                            continue
                        content = f.read().decode("utf-8", errors="ignore")
                        short_name = f"{archive_name}/{os.path.basename(member.name)}"
                        seen = set()
                        for line in content.splitlines():
                            cleaned = clean_line(line)
                            if cleaned and len(cleaned) >= 10 and is_important(cleaned) and cleaned not in seen:
                                seen.add(cleaned)
                                results.append((short_name, cleaned))
                    except Exception:
                        continue
        elif archive_path.endswith(".zip"):
            with zipfile.ZipFile(archive_path, "r") as zf:
                for name in zf.namelist():
                    if not any(p in name.lower() for p in ARCHIVE_IMPORTANT_PATTERNS):
                        continue
                    info = zf.getinfo(name)
                    if info.file_size > 50 * 1024 * 1024:
                        continue
                    try:
                        content = zf.read(name).decode("utf-8", errors="ignore")
                        short_name = f"{archive_name}/{os.path.basename(name)}"
                        seen = set()
                        for line in content.splitlines():
                            cleaned = clean_line(line)
                            if cleaned and len(cleaned) >= 10 and is_important(cleaned) and cleaned not in seen:
                                seen.add(cleaned)
                                results.append((short_name, cleaned))
                    except Exception:
                        continue
    except Exception as e:
        print(f"  Warning: Could not extract {archive_name}: {e}")
    return results

def scan_log_folder(folder):
    """Scan folder recursively for all log files and archives."""
    all_logs = []
    for root, dirs, files in os.walk(folder):
        for file in sorted(files):
            filepath = os.path.join(root, file)
            rel_path = os.path.relpath(filepath, folder)
            if file.endswith(SUPPORTED_EXTENSIONS):
                entries = read_filtered_logs(filepath, rel_path)
                if entries:
                    print(f"  {rel_path}: {len(entries)} important entries")
                all_logs.extend(entries)
            elif file.endswith(ARCHIVE_EXTENSIONS):
                entries = extract_and_read_archive(filepath, rel_path)
                if entries:
                    print(f"  {rel_path} (archive): {len(entries)} important entries")
                all_logs.extend(entries)
    return all_logs

# Scan the log folder
print(f"Scanning {LOG_FOLDER}/ for log files...\n")
logs = scan_log_folder(LOG_FOLDER)
print(f"\nTotal: {len(logs)} important log entries from {len(set(f for f, _ in logs))} files")

## Step 5: Preview Loaded Logs with Severity

Let's see what we loaded — each log line gets a severity tag (ERROR, WARNING, FAIL, INFO).

In [ ]:
# Show severity breakdown
severity_counts = {"ERROR": 0, "FAIL": 0, "WARNING": 0, "INFO": 0}
for _, msg in logs:
    sev = detect_severity(msg)
    severity_counts[sev] += 1

print("Severity Breakdown:")
for sev, count in severity_counts.items():
    print(f"  {sev}: {count}")

# Show all log entries with severity
print(f"\nAll {len(logs)} log entries:")
print("-" * 80)
for filename, msg in logs:
    sev = detect_severity(msg)
    print(f"[{sev:7s}] [{filename}] {msg}")

## Step 6: Create Embeddings with Sentence Transformers

Convert each log line into a **384-dimensional vector** using the `all-MiniLM-L6-v2` model. These vectors capture the meaning of each log line.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the embedding model
print("Loading embedding model (all-MiniLM-L6-v2)...")
model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert all log messages to vectors
texts = [msg for _, msg in logs]
print(f"Encoding {len(texts)} log entries...")
embeddings = model.encode(texts, show_progress_bar=True, batch_size=256)
embeddings = np.array(embeddings).astype("float32")

print(f"\nDone! Shape: {embeddings.shape}")
print(f"Each log line is now a {embeddings.shape[1]}-dimensional vector")

## Step 7: Build FAISS Vector Store

Store all embeddings in a **FAISS index** for fast similarity search. Also save to disk so we don't have to rebuild every time.

In [ ]:
import faiss
import pickle

VECTOR_DIR = "vector_store"
INDEX_FILE = os.path.join(VECTOR_DIR, "faiss_index.bin")
LOG_FILE = os.path.join(VECTOR_DIR, "logs.pkl")
os.makedirs(VECTOR_DIR, exist_ok=True)

# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# Save to disk
faiss.write_index(index, INDEX_FILE)
with open(LOG_FILE, "wb") as f:
    pickle.dump(logs, f)

print(f"Vector store built: {index.ntotal} vectors saved to {VECTOR_DIR}/")
print(f"Vector dimension: {dimension}")

## Step 8: Search Similar Logs (Vector Retrieval)

Ask a question in plain English → the system finds the most relevant log entries using vector similarity search.

In [ ]:
def search_logs(query, top_k=5):
    """Search for log entries most similar to the query."""
    query_vector = model.encode([query])
    distances, indices = index.search(query_vector, min(top_k, len(logs)))
    
    results = []
    print(f"Query: '{query}'\n")
    print(f"Top {top_k} most relevant logs:")
    print("-" * 80)
    for i in range(min(top_k, len(logs))):
        idx = indices[0][i]
        dist = distances[0][i]
        filename, msg = logs[idx]
        sev = detect_severity(msg)
        results.append((filename, msg, dist))
        print(f"  [{sev}] [{filename}] {msg}  (distance: {dist:.4f})")
    return results

# Try searching!
search_logs("What errors occurred?")
print()
search_logs("network timeout failure")

## Step 9: LLM Analysis — Send Logs to Groq for Root Cause

Now we combine **retrieval** (FAISS search) + **generation** (Groq LLM) = **RAG**.

The LLM reads the retrieved logs and gives a root cause analysis.

In [ ]:
def ask_llm(messages, max_tokens=500):
    """Send messages to Groq LLM and get response."""
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            max_tokens=max_tokens
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"AI Error: {e}"

def analyze_with_rag(query, top_k=5):
    """Full RAG: retrieve relevant logs → send to LLM → get analysis."""
    # Step 1: Retrieve relevant logs
    query_vector = model.encode([query])
    distances, indices = index.search(query_vector, min(top_k, len(logs)))
    
    context = ""
    for i in range(min(top_k, len(logs))):
        filename, msg = logs[indices[0][i]]
        context += f"[{filename}] {msg}\n"
    
    # Step 2: Send to LLM
    messages = [
        {
            "role": "system",
            "content": (
                "You are a telecom log analysis assistant. "
                "Analyze the provided log entries and give:\n"
                "- Root Cause (one-line summary)\n"
                "- Severity (CRITICAL / HIGH / MEDIUM / LOW)\n"
                "- Details (detailed explanation)\n"
                "- Recommendation (what to do next)"
            )
        },
        {
            "role": "user",
            "content": f"Question: {query}\n\nRelevant log entries:\n{context}"
        }
    ]
    
    return ask_llm(messages)

# Test RAG analysis
print("=" * 80)
print("RAG Analysis: Analyzing all errors")
print("=" * 80)
result = analyze_with_rag("Analyze all errors and find root cause")
print(result)

## Step 10: Iterative AI Investigation (Agent Mode)

This is the **agent-style** analysis — same as our full project. The AI:
1. Reads initial errors
2. Asks which files to check next
3. We provide those logs
4. AI digs deeper
5. Repeats until root cause is found

In [ ]:
def get_logs_from_file(file_pattern):
    """Get all log entries matching a file pattern."""
    pattern_lower = file_pattern.lower()
    return [(f, m) for f, m in logs if pattern_lower in f.lower()]

def iterative_investigation(query="Analyze all errors and find root cause"):
    """Run the full iterative AI investigation — same as rag_system.py"""
    
    # Get list of available files
    available_files = sorted(set(f for f, _ in logs))
    file_list = "\n".join(f"  - {f}" for f in available_files)
    
    # Build initial error summary
    error_summary = ""
    for fname, msg in logs[:20]:
        sev = detect_severity(msg)
        error_summary += f"  - [{sev}] [{fname}] {msg}\n"
    
    # Step 1: Initial diagnosis
    print("=" * 80)
    print("STEP 1: Initial Scan — Sending errors to AI")
    print("=" * 80)
    
    messages = [
        {
            "role": "system",
            "content": (
                "You are a telecom log debugging assistant. "
                "The user will give you initial error logs and a list of available files. "
                "First give a brief initial diagnosis. "
                "Then tell which file(s) to check next for deeper debugging. "
                "Reply with EXACTLY this format at the end:\n"
                "CHECK_NEXT: filename1, filename2\n"
                "Or if no more files needed:\n"
                "CHECK_NEXT: DONE"
            )
        },
        {
            "role": "user",
            "content": (
                f"User question: {query}\n\n"
                f"Initial error scan:\n{error_summary}\n\n"
                f"Available files:\n{file_list}"
            )
        }
    ]
    
    response = ask_llm(messages)
    print("\nAI Initial Diagnosis:")
    print(response)
    messages.append({"role": "assistant", "content": response})
    
    # Step 2+: Iterative deep-dive
    MAX_ITERATIONS = 3
    for iteration in range(MAX_ITERATIONS):
        # Parse CHECK_NEXT
        check_next = None
        for line in response.splitlines():
            if line.strip().upper().startswith("CHECK_NEXT:"):
                check_next = line.split(":", 1)[1].strip()
                break
        
        if not check_next or check_next.upper() == "DONE":
            print(f"\n{'=' * 80}")
            print("AI investigation complete!")
            print("=" * 80)
            break
        
        requested_files = [f.strip() for f in check_next.split(",")]
        print(f"\n{'=' * 80}")
        print(f"STEP {iteration + 2}: AI requested files: {requested_files}")
        print("=" * 80)
        
        # Gather logs from requested files
        deep_logs = ""
        for pattern in requested_files:
            matched = get_logs_from_file(pattern)
            if matched:
                deep_logs += f"\n[{pattern}] ({len(matched)} entries):\n"
                for fname, msg in matched[:20]:
                    deep_logs += f"  - {msg}\n"
            else:
                deep_logs += f"\n[{pattern}]: No matching file found.\n"
        
        messages.append({
            "role": "user",
            "content": (
                f"Here are the logs from the files you requested:\n{deep_logs}\n\n"
                "Analyze these and give deeper root cause analysis. "
                "If you need more files, reply with CHECK_NEXT: filename1, filename2\n"
                "If investigation is complete, reply with CHECK_NEXT: DONE"
            )
        })
        
        response = ask_llm(messages)
        print("\nAI Deep Analysis:")
        print(response)
        messages.append({"role": "assistant", "content": response})

# Run the investigation!
iterative_investigation()

## Step 11: Upload and Analyze New Log Files

Upload new log files directly into the notebook — supports all file types (.txt, .log, .json, .csv, .xml, .html, .tgz, .zip). The system will parse, embed, and analyze them.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

output_area = widgets.Output()

def process_uploaded_file(change):
    """Process files uploaded through the widget."""
    global logs, embeddings, index
    
    with output_area:
        clear_output()
        for uploaded_file in uploader.value:
            filename = uploaded_file.name
            content_bytes = uploaded_file.content
            print(f"\nProcessing: {filename}")
            
            new_entries = []
            
            # Handle archives
            if filename.lower().endswith(ARCHIVE_EXTENSIONS):
                try:
                    if filename.endswith((".tgz", ".tar.gz")):
                        with tarfile.open(fileobj=io.BytesIO(content_bytes), mode="r:gz") as tar:
                            for member in tar.getmembers():
                                if not member.isfile() or member.size > 50*1024*1024:
                                    continue
                                if not any(p in member.name.lower() for p in ARCHIVE_IMPORTANT_PATTERNS):
                                    continue
                                f = tar.extractfile(member)
                                if f is None:
                                    continue
                                text = f.read().decode("utf-8", errors="ignore")
                                short = f"{filename}/{os.path.basename(member.name)}"
                                seen = set()
                                for line in text.splitlines():
                                    c = clean_line(line)
                                    if c and len(c) >= 10 and is_important(c) and c not in seen:
                                        seen.add(c)
                                        new_entries.append((short, c))
                    elif filename.endswith(".zip"):
                        with zipfile.ZipFile(io.BytesIO(content_bytes), "r") as zf:
                            for name in zf.namelist():
                                if not any(p in name.lower() for p in ARCHIVE_IMPORTANT_PATTERNS):
                                    continue
                                info = zf.getinfo(name)
                                if info.file_size > 50*1024*1024:
                                    continue
                                text = zf.read(name).decode("utf-8", errors="ignore")
                                short = f"{filename}/{os.path.basename(name)}"
                                seen = set()
                                for line in text.splitlines():
                                    c = clean_line(line)
                                    if c and len(c) >= 10 and is_important(c) and c not in seen:
                                        seen.add(c)
                                        new_entries.append((short, c))
                except Exception as e:
                    print(f"  Error extracting archive: {e}")
            else:
                # Text-based files
                try:
                    content = content_bytes.decode("utf-8", errors="ignore")
                except Exception:
                    content = content_bytes.decode("latin-1", errors="ignore")
                
                seen = set()
                for line in content.splitlines():
                    cleaned = clean_line(line)
                    if cleaned and len(cleaned) >= 10 and is_important(cleaned) and cleaned not in seen:
                        seen.add(cleaned)
                        new_entries.append((filename, cleaned))
            
            if new_entries:
                # Add to logs and update vector store
                new_texts = [msg for _, msg in new_entries]
                new_embeddings = model.encode(new_texts, batch_size=256)
                new_embeddings = np.array(new_embeddings).astype("float32")
                index.add(new_embeddings)
                logs.extend(new_entries)
                
                print(f"  Added {len(new_entries)} entries to vector store")
                print(f"  Total vectors now: {index.ntotal}")
                
                # Show severity breakdown for new file
                for fname, msg in new_entries[:10]:
                    sev = detect_severity(msg)
                    print(f"    [{sev}] {msg}")
                if len(new_entries) > 10:
                    print(f"    ... and {len(new_entries) - 10} more")
            else:
                print(f"  No important entries found in {filename}")

# Create upload widget
uploader = widgets.FileUpload(
    accept='.txt,.log,.json,.csv,.xml,.html,.htm,.cfg,.tgz,.gz,.zip',
    multiple=True,
    description='Upload Logs'
)
uploader.observe(process_uploaded_file, names='value')

print("Upload log files here (they will be auto-analyzed):")
display(uploader)
display(output_area)

In [ ]:
def compare_pass_fail(pass_path, fail_path):
    """Compare PASS and FAIL log files to find the real failure cause."""
    
    # Read both files
    with open(pass_path, "r", encoding="utf-8", errors="ignore") as f:
        pass_content = f.read()
    with open(fail_path, "r", encoding="utf-8", errors="ignore") as f:
        fail_content = f.read()
    
    pass_name = os.path.basename(pass_path)
    fail_name = os.path.basename(fail_path)
    
    # Parse all lines
    pass_lines = set()
    for line in pass_content.splitlines():
        cleaned = clean_line(line)
        if cleaned and len(cleaned) >= 10:
            pass_lines.add(cleaned)
    
    fail_lines = []
    fail_set = set()
    for line in fail_content.splitlines():
        cleaned = clean_line(line)
        if cleaned and len(cleaned) >= 10 and cleaned not in fail_set:
            fail_set.add(cleaned)
            fail_lines.append(cleaned)
    
    # Classify
    common_errors = []
    fail_only_errors = []
    
    for line in fail_lines:
        is_in_pass = line in pass_lines
        is_err = is_important(line)
        if is_err and is_in_pass:
            common_errors.append(line)
        elif is_err and not is_in_pass:
            fail_only_errors.append(line)
    
    print(f"PASS log: {pass_name} ({len(pass_lines)} unique lines)")
    print(f"FAIL log: {fail_name} ({len(fail_set)} unique lines)")
    print(f"\nCommon errors (ignorable): {len(common_errors)}")
    print(f"FAIL-only errors (critical): {len(fail_only_errors)}")
    
    if common_errors:
        print(f"\nIgnorable errors (in BOTH logs):")
        for line in common_errors[:10]:
            print(f"  ⚪ {line}")
    
    if fail_only_errors:
        print(f"\nCritical errors (ONLY in FAIL):")
        for line in fail_only_errors[:15]:
            print(f"  🔴 {line}")
    
    # Ask LLM for analysis
    common_summary = "\n".join(f"  - {l}" for l in common_errors[:15]) or "  (none)"
    fail_only_summary = "\n".join(f"  - {l}" for l in fail_only_errors[:20]) or "  (none)"
    
    messages = [
        {
            "role": "system",
            "content": (
                "You are a telecom log analyst. Compare PASS and FAIL logs. "
                "Focus on errors ONLY in the FAIL log — those caused the failure. "
                "Ignore common errors (present in both). "
                "Give: Root Cause, Severity, Details, Recommendation."
            )
        },
        {
            "role": "user",
            "content": (
                f"PASS: {pass_name} ({len(pass_lines)} lines)\n"
                f"FAIL: {fail_name} ({len(fail_set)} lines)\n\n"
                f"Errors in BOTH (ignorable):\n{common_summary}\n\n"
                f"Errors ONLY in FAIL (critical):\n{fail_only_summary}"
            )
        }
    ]
    
    print(f"\n{'=' * 80}")
    print("AI Comparison Analysis:")
    print("=" * 80)
    analysis = ask_llm(messages, max_tokens=800)
    print(analysis)

# Example usage (update paths to your actual PASS and FAIL logs):
# compare_pass_fail("data/logs/pass_log.txt", "data/logs/fail_log.txt")
print("To compare logs, call: compare_pass_fail('path/to/pass.log', 'path/to/fail.log')")

## Step 13: Ask Your Own Questions

Use this cell to ask any question about the loaded logs. The RAG system will retrieve relevant entries and use the LLM to answer.

In [ ]:
# Change the question below and re-run this cell!
your_question = "What caused the test failure?"

print(f"Question: {your_question}\n")
print("=" * 80)
answer = analyze_with_rag(your_question)
print(answer)